## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting?
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Shravya Etta

**Date:** 09/29/2026

**QUESTION 1**

1. A regression predicts a numerical outcome along an interval, such as blood pressure, temperature, etc. A classification predicts a discrete category label or class type, such as healthy vs. diseased cell types. Classes are a distict category without a continous numerical distance relating them.

2. A confusion table is an N x N contingency matrix that cross-tabulates a classifier's predicted labels against known labels. In binary classification, a confusion table separates outcomes into four possibilities:
* True positives and true negatives are correctly identified predictions.
* False positives are a type I error, meaning that actual negative values are predicted as positive.
* False negatives are a type II error, where actual positive values are predicted as negative.

  It helps us undersand a model's performance by identifying whether errors are skewed towards missing possitive cases (false negatives) or false positives. Additionally, the raw counts needed to calculate metrics like sensitivity, specificity, and precision are provided by a confusion matrix.

3. The SSE measures the total squared discrepancy between observed values and model-predicted values, meaning that it captures the portion of variance that the model can't explain. Squaring the residuals, specifically, causes a larger magnitude of "penalty" to larger deviations, which makes SSE sensitive to extreme errors in prediction. Hence, minimizing SSE is a good criteria when finding optimal parameter values/weights.

4. Overfitting is when the model momorizes details of the training set, random noise, and variance. This results in very low training error, but significantly higher error on unseen and testing data.

    Underfitting is when the model is too simple to be able to understand the true trends in the data, resulting in high error on training data as well as on unseen and testing data (ie. high bias).

5. Splitting data into training and testing sets is important because evaluating a model's performance solely on the training set causes bias. A separate testing set helps simulate real-world generalization to previously unseen data inputs.

    For k-Nearest Neighbors (kNN), a k value of 1 results in zero training error but creates noisy deciison boundaries, indicates high variance and overfitting. High k values, meanwhile, oversmooths boundaries and indicates high bias and underfitting. Selecting the k value based on the final testing data could cause overfitting to the parameters in that specific testing sample. Hence, k value selection can improve model performance when based on a training/testing split, as well as reserving the final testing set to be used as a final evalulation metric.

6. A strength of reporting a class label as a prediction is that it's more direct and unambigous, which is helpful in automatated processes with downstream logic. It also requires no further threshold interpretation by the user and has a low computational need in terms of output storage and parsing. A weakness, however, is that disregards prediction confidence, meaning that confidences of 57% and 90% would be treated equally. It is also more rigid in nature, as it hardcodes a default threshold value (ex. 0.5) that can't adapt to changes in false positive or false negative erros.

    Probability distributions preserve model confidence and uncertainty, allow for dynamic thresholding, and allow for ranking instances and calibrated decision making. It's weaknesses are that probability could be uncalibrated, such as naive Bayes models that can result in unscaled confidence scores. Another weakness is that it requires establishing a threshold value before starting an automated process.

**QUESTION 2**

In [3]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("land_mines.csv")

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")

print(df.describe().round(4), "\n")

print(f"Target distribution: {df["mine_type"].value_counts().sort_index()}", "\n")

print(f"Mean feature values by mine type:\n {df.groupby("mine_type").mean().round(4)}", "\n")


Rows: 338, Columns: 4

        voltage    height      soil  mine_type
count  338.0000  338.0000  338.0000   338.0000
mean     0.4306    0.5089    0.5036     2.9527
std      0.1958    0.3060    0.3442     1.4197
min      0.1977    0.0000    0.0000     1.0000
25%      0.3097    0.2727    0.2000     2.0000
50%      0.3595    0.5455    0.6000     3.0000
75%      0.4826    0.7273    0.8000     4.0000
max      1.0000    1.0000    1.0000     5.0000 

Target distribution: mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64 

Mean feature values by mine type:
            voltage  height    soil
mine_type                         
1           0.2965  0.4955  0.4930
2           0.7211  0.4870  0.5086
3           0.4024  0.5275  0.4970
4           0.3454  0.5069  0.5030
5           0.3796  0.5301  0.5169 



In [4]:
X = df[["voltage", "height", "soil"]]
Y = df["mine_type"]

# 50:50 training:testing split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.50, random_state=42, stratify=Y
)

# standardizing features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
k_vals = range(1, 26)
cv_scores = []

for k in k_vals:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_scaled, Y_train, cv=cv, scoring="accuracy")
    cv_scores.append(scores.mean())

k_optimal = k_vals[np.argmax(cv_scores)]
print(f"Optimal k: {k_optimal}; CV Accuracy: {max(cv_scores):.4f}\n")

# training final model on entire training set
model = KNeighborsClassifier(n_neighbors=k_optimal)
model.fit(X_train_scaled, Y_train)

Optimal k: 6; CV Accuracy: 0.4975



KNeighborsClassifier(n_neighbors=6)

I selected k using a 5-fold stratified cross-validation over the range [1,25]. The cross-validation curve peaks at k-6, indicating that is the most optimal k value in regards to balance variance and bias.

In [6]:
Y_pred = model.predict(X_test_scaled)

conf_matrix = confusion_matrix(Y_test, Y_pred)
labels = [f"Type {i}" for i in range(1, 6)]
conf_df = pd.DataFrame(conf_matrix, index=[f"Actual {l}" for l in labels], columns=[f"Pred {l}" for l in labels])

print(f"Confusion Matrix\n {conf_df}\n")
print(f"Classification Report\n {classification_report(Y_test, Y_pred, target_names=labels)}")

Confusion Matrix
                Pred Type 1  Pred Type 2  Pred Type 3  Pred Type 4  Pred Type 5
Actual Type 1           23            0            7            5            1
Actual Type 2            0           34            0            1            0
Actual Type 3            9            3            5            6           10
Actual Type 4           15            5            7            3            3
Actual Type 5            9            2           10            6            5

Classification Report
               precision    recall  f1-score   support

      Type 1       0.41      0.64      0.50        36
      Type 2       0.77      0.97      0.86        35
      Type 3       0.17      0.15      0.16        33
      Type 4       0.14      0.09      0.11        33
      Type 5       0.26      0.16      0.20        32

    accuracy                           0.41       169
   macro avg       0.35      0.40      0.37       169
weighted avg       0.36      0.41      0.38       

To better analyze the accuracy of the model, I also added a classification report to convert the raw counts in the confusion matrix into more interpretable metrics.

There is an overall accuracy of roughly 41% on the test set. The model best succeeds with the type 2 prediction, as seen by its 97% recall and 77% precision. The model fails in the fact that classes 3, 4, and 5 have overlaps in features. The type 4 prediction has a low recall of 9%, and type 5 also has a low recall of 16%. Furthermore, over half of the Type 3, 4, and 5 mines are incorrectly classified as Type 1.

This model could not be used as a decision-maker on its own, but its outputs could be treated as an alert as oposed to a confirmation. Users could also associate certain predictions with certain risk tiers. For instance, when the model predicts Type 2, the mine is likely to actually be Type 2 and have a high voltage signature. If the model predicts Types 1, 2, 3, or 5, the system should flag the mine instead of giving it a definitive label. This way, field workers can assume the most sensitive/dangerous possible detonation type.

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [To be provided]
- **AI tool and model used:** [To be provided]

### *Prompts used

1. [Paste prompt]
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]